# Introduction

Word2Vec is a word embedding technique in NLP that represents word as vector in a continuous space.

- Converts words into numerical vector for ML models
- Captures semantic relationship between words
- Word with similar meaning have similar vector representation
- Uses two main architectures
    - Contineous Bag of Words
    - Skip Gram

### Continuous Bag of Words (CBOW)

The CBOW model predicts the current word given context words within a specific window.

### Skip Gram

The skip gram predicts the surrounding context words within specific window given current word.

In [ ]:
!pip install gensim
!pip install python-levenshtein

In [2]:
import gensim # NLP library for Word2Vec
import pandas as pd
from urllib.request import urlretrieve
import gzip
import shutil

### **Reading and Exploring the Dataset**

The datasent we are using here is a s subset of Amazon reviews from the cell phones & Accessories category The data is stored as JSON file and can be read using pandas.

Link to the Dataset:  http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Cell_Phones_and_Accessories_5.json.gz

In [3]:
url = "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Cell_Phones_and_Accessories_5.json.gz"
gz_filename = "reviews_Cell_Phones_and_Accessories_5.json.gz"
json_filename = "reviews_Cell_Phones_and_Accessories_5.json"

urlretrieve(url, gz_filename)

with gzip.open(gz_filename, "rb") as compressed_file:
    with open(json_filename, "wb") as extracted_file:
        shutil.copyfileobj(compressed_file, extracted_file)

reviews = pd.read_json(json_filename, lines=True)

reviews.head()


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A30TL5EWN6DFXT,120401325X,christina,"[0, 0]",They look good and stick good! I just don't li...,4,Looks Good,1400630400,"05 21, 2014"
1,ASY55RVNIL0UD,120401325X,emily l.,"[0, 0]",These stickers work like the review says they ...,5,Really great product.,1389657600,"01 14, 2014"
2,A2TMXE2AFO7ONB,120401325X,Erica,"[0, 0]",These are awesome and make my phone look so st...,5,LOVE LOVE LOVE,1403740800,"06 26, 2014"
3,AWJ0WZQYMYFQ4,120401325X,JM,"[4, 4]",Item arrived in great time and was in perfect ...,4,Cute!,1382313600,"10 21, 2013"
4,ATX7CZYFXI1KW,120401325X,patrice m rogoza,"[2, 3]","awesome! stays on, and looks great. can be use...",5,leopard home button sticker for iphone 4s,1359849600,"02 3, 2013"


In [4]:
reviews.shape

(194439, 9)

We are interested only in reviewText column, but we need to do some preprocessing to train Word2Vec model.
- Convert all word to lower case
- Remove puncutaions
- Remove words like a, I, the, it, etc.


### Lets take a look how this preprocessing can be done

In [5]:
reviews['reviewText'][0]

"They look good and stick good! I just don't like the rounded shape because I was always bumping it and Siri kept popping up and it was irritating. I just won't buy a product like this again"

In [6]:
gensim.utils.simple_preprocess(reviews['reviewText'][0])

['they',
 'look',
 'good',
 'and',
 'stick',
 'good',
 'just',
 'don',
 'like',
 'the',
 'rounded',
 'shape',
 'because',
 'was',
 'always',
 'bumping',
 'it',
 'and',
 'siri',
 'kept',
 'popping',
 'up',
 'and',
 'it',
 'was',
 'irritating',
 'just',
 'won',
 'buy',
 'product',
 'like',
 'this',
 'again']

You can see it has broken the sentence into list of tokens and applied the preprocessing mentioned above

Now, let's do the same for entire reviewText column

In [7]:
review_text = reviews['reviewText'].apply(gensim.utils.simple_preprocess)
review_text

,reviewText
0,"[they, look, good, and, stick, good, just, don..."
1,"[these, stickers, work, like, the, review, say..."
2,"[these, are, awesome, and, make, my, phone, lo..."
3,"[item, arrived, in, great, time, and, was, in,..."
4,"[awesome, stays, on, and, looks, great, can, b..."
...,...
194434,"[works, great, just, like, my, original, one, ..."
194435,"[great, product, great, packaging, high, quali..."
194436,"[this, is, great, cable, just, as, good, as, t..."
194437,"[really, like, it, becasue, it, works, well, w..."


In [8]:
model = gensim.models.Word2Vec(
    window=10, # Maximum distance between the current and predicted word within a sentence.
    min_count=2, # Minimum frequency of a word to be considered in the model.
    workers=4 # Number of CPU cores to use for training.
)

In [9]:
model.build_vocab(review_text, progress_per=1000)

In [10]:
print(model.epochs)
print(model.corpus_count)

5
194439


In [11]:
model.train(review_text, total_examples=model.corpus_count, epochs=model.epochs)

(61514058, 83868975)

In [12]:
from google.colab import files

model_path = "/content/word2vec-amazon-cell-accessories-reviews-short.model"
model.save(model_path)
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
model.wv.most_similar("bad", topn=10)

[('shabby', 0.6743184924125671),
 ('terrible', 0.6701129078865051),
 ('horrible', 0.6095079779624939),
 ('crappy', 0.5798107385635376),
 ('good', 0.5767238140106201),
 ('awful', 0.5745713114738464),
 ('poor', 0.5274395942687988),
 ('okay', 0.5268374681472778),
 ('badly', 0.5222069621086121),
 ('ok', 0.5102858543395996)]

In [19]:
# Cosine similarity between two words
model.wv.similarity(w1 = "great", w2 = "awesome")

np.float32(0.7588106)

### Exercise
Train a word2vec model on the [Sports & Outdoors Reviews Dataset](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz) Once you train a model on this, find the words most similar to 'awful' and find similarities between the following word tuples: ('good', 'great'), ('slow','steady')

